# Climate indices — quickstart (live)

The `climate-indices` backend fetches monthly **teleconnection index**
series (ENSO/ONI, NAO, AO, PDO, AMO, SOI, PNA, …) from two open ASCII
sources — **NOAA PSL** and the **KNMI Climate Explorer** — and returns
them as a tidy long-format `pandas.DataFrame` with columns `date`,
`index`, `value`, `source`.

These indices are **global scalar monthly series**: one number per
month, no geometry. So unlike the raster backends there is no bounding
box and no grid — you ask for index ids and a date window, and you get
back a time series.

This notebook fetches data **live** from both sources.

In [ ]:
import matplotlib.pyplot as plt

from earthlens.earthlens import EarthLens

## A single index (NOAA PSL)

Pass the index id(s) in `variables=` and a `[start, end]` window. The
Oceanic Niño Index (`oni`) is the canonical ENSO indicator — a 3-month
running mean of the Niño 3.4 sea-surface-temperature anomaly.

In [ ]:
oni = EarthLens(
    data_source='climate-indices',
    variables=['oni'],
    start='1990-01-01',
    end='2020-12-31',
    path='ci_out',
).download()

oni.head()

Every row is one month (`date` is the first of the month). The
missing-value sentinel from the source file is mapped to `NaN` (kept,
not dropped, so gaps stay visible). `download()` also wrote the table to
a CSV under `path`.

In [ ]:
print('rows:', len(oni))
print('span:', oni['date'].min().date(), '->', oni['date'].max().date())
print('source:', oni['source'].unique().tolist())
oni['value'].describe()[['min', 'mean', 'max']]

## Several indices, two sources at once

Ask for more than one id and the result concatenates them, told apart by
the `index` (and `source`) column. Here `oni` and `nao` come from NOAA
PSL while `amo` (the detrended Atlantic Multidecadal Oscillation) comes
from the KNMI Climate Explorer — two different ASCII dialects, one tidy
frame.

In [ ]:
df = EarthLens(
    data_source='climate-indices',
    variables=['oni', 'nao', 'amo'],
    start='1980-01-01',
    end='2020-12-31',
    path='ci_out',
).download()

df.groupby('index').agg(
    rows=('value', 'size'),
    source=('source', 'first'),
    first=('date', 'min'),
    last=('date', 'max'),
)

## Pivot to wide form and plot

The long frame pivots to a month × index table in one call — handy for
plotting or correlating indices against each other.

In [ ]:
wide = df.pivot(index='date', columns='index', values='value')
wide.tail()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4))
ax.axhline(0.0, color='0.6', lw=0.8)
ax.fill_between(
    wide.index,
    0,
    wide['oni'].clip(lower=0),
    color='tab:red',
    alpha=0.6,
    label='El Nino (ONI > 0)',
)
ax.fill_between(
    wide.index,
    0,
    wide['oni'].clip(upper=0),
    color='tab:blue',
    alpha=0.6,
    label='La Nina (ONI < 0)',
)
ax.set_title('Oceanic Nino Index (ONI), monthly')
ax.set_ylabel('degC anomaly')
ax.legend(loc='upper left')
fig.tight_layout()

## Roll up to annual means

There is no server-side aggregation for these scalar series (and
`download(aggregate=...)` is rejected — see the catalog/behaviour
notebook). Any rollup is a one-liner on the returned frame:

In [ ]:
annual = (
    df.assign(year=df['date'].dt.year)
    .groupby(['index', 'year'])['value']
    .mean()
    .unstack('index')
)
annual.tail()

## Takeaway

- `EarthLens(data_source='climate-indices', variables=[...], start=...,
  end=...)` returns a long `date / index / value / source` frame.
- Multiple indices — even across the two sources — come back in one
  frame, distinguished by `index` / `source`.
- The result is plain pandas: pivot, plot, resample, correlate as you
  like. See the [catalog & behaviour](02_catalog_and_behavior.ipynb)
  notebook for the shipped index list and the no-bbox / no-aggregate
  design.